# Extension 3 -- NFHS-4 to NFHS-5 Doubly Robust Temporal Comparison

**Research question.** Did the adjusted private-vs-public Cesarean risk difference change between NFHS-4 (2015-16) and NFHS-5 (2019-21)?

**Relationship to the frozen V2 pipeline.** This notebook does not modify anything under `notebooks/v2/` or `outputs/`. NFHS-5 records reuse `data/processed/df_model_v2.parquet` exactly as loaded in `notebooks/v2/06-07`. NFHS-4 records require a separately harmonized local file (`extensions_work/shared/config.NFHS4_PROCESSED_DATA_PATH`) that this notebook does not build automatically -- see Section 3 for the harmonization helper and its explicit "verify against the NFHS-4 recode manual" caveats.

**Method, stated up front.** Round-specific cross-fitted AIPW (identical machinery to `notebooks/v2/07`) is run independently within each survey round, using an identical, pre-harmonized confounder set across rounds. The *change* in the adjusted risk difference (NFHS-5 minus NFHS-4) is estimated with a **directly computed confidence interval**, not by subtracting two independently reported point estimates: because the two survey rounds are statistically independent samples (no respondent overlap between waves), the joint sampling distribution of `(RD_hat_4, RD_hat_5)` is obtained by resampling PSUs independently within each round's own bootstrap loop and pairing same-index replicates (`change_b = RD5_b - RD4_b`), which yields a formally valid percentile CI for the change itself -- see Section 9 for why this is correct.

**Interpretation rule, repeated at the end:** a change (or lack of change) in the adjusted gap over time is **not**, by itself, causal evidence of a specific policy or programmatic effect. It describes how the *adjusted association* moved between two cross-sectional survey waves under each round's own measured-confounding assumption.

**Not run here.** Written but not executed -- this sandbox has no NFHS data. Run it locally once both `data/processed/df_model_v2.parquet` and a harmonized `df_model_nfhs4.parquet` exist.

## 1. Imports, shared config/utils

In [1]:
import sys
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

sys.path.append(str(Path("../shared").resolve()))
import config
import utils

pd.set_option("display.max_columns", None)

OUTPUTS_DIR = config.EXT03_OUTPUTS_DIR
RANDOM_STATE = config.RANDOM_STATE
N_FOLDS = config.N_FOLDS
N_BOOTSTRAP = config.N_BOOTSTRAP

## 2. Variable crosswalk -- built and saved BEFORE any modeling (handoff step 22)

The crosswalk below is the single source of truth for what is harmonized, from where, and with what caveat. `verified` is `False` for any mapping this notebook cannot confirm without the NFHS-4 recode manual in hand -- those variables are dropped from the reduced/sensitivity confounder set (Section 5), not silently trusted.

In [2]:
crosswalk_rows = [
    ("facility_type", "exposure", "m15 (recoded: 10-13 home, 20-27 public, 30-33 private, else other)",
     "m15 (same recode function)", True,
     "DHS place-of-delivery coding is stable across DHS-7-generation surveys, but the exact 96/other "
     "boundary should still be spot-checked against the NFHS-4 recode manual before trusting silently."),
    ("csection", "outcome", "m17", "m17", True, "Direct match."),
    ("respondent_id", "design", "caseid", "caseid", True, "Round-specific identifier; never pooled as if the same respondent."),
    ("cluster_number", "design", "v001", "v001", True, "Round-specific PSU; bootstrap resampling stays within round."),
    ("sample_stratum_v022", "design", "v022", "v022", True, "Round-specific sampling stratum."),
    ("sample_weight_normalized", "design", "v005 / 1e6", "v005 / 1e6", True, "Each round uses its own round-specific weight; never mixed across rounds."),
    ("birth_order", "confounder", "bord", "bord", True, "Direct match."),
    ("twin_order", "confounder", "b0", "b0", True, "Direct match."),
    ("wealth_index", "confounder", "v190", "v190", True,
     "Direct match, but DHS wealth-index construction methodology has known cross-round revisions -- "
     "treat cross-round wealth_index comparisons as ordinal within-round, not as a continuously "
     "comparable scale across rounds."),
    ("education_years", "confounder", "v133 (97 -> missing)", "v133 (97 -> missing)", True, "Direct match."),
    ("residence", "confounder", "v025", "v025", True, "Direct match."),
    ("religion", "confounder", "v130", "v130", True, "Direct match."),
    ("social_group", "confounder", "s116 (8 -> missing)", "s116 (8 -> missing)", False,
     "NOT YET VERIFIED for NFHS-4: confirm s116 exists identically in the NFHS-4 Birth Recode before "
     "treating it as harmonized. If absent or differently coded, it is dropped from the reduced/"
     "sensitivity confounder set below."),
    ("state", "confounder", "v024", "v024", False,
     "State/UT boundary changes between 2015-16 and 2019-21 (e.g. new union territories) require an "
     "explicit crosswalk of state codes before pooling -- NOT built automatically here; verify locally "
     "before trusting a pooled state effect."),
    ("survey_round", "design (derived)", "n/a", "n/a", True, "New column: 'NFHS-4' / 'NFHS-5', the time indicator for the pooled comparison."),
]

variable_crosswalk = pd.DataFrame(
    crosswalk_rows,
    columns=["harmonized_name", "role", "nfhs5_source", "nfhs4_source", "verified", "harmonization_note"],
)
variable_crosswalk.to_csv(OUTPUTS_DIR / "nfhs4_nfhs5_variable_crosswalk.csv", index=False)
print("Saved:", OUTPUTS_DIR / "nfhs4_nfhs5_variable_crosswalk.csv")
variable_crosswalk

Saved: C:\Users\rifan\Downloads\IPD_SEM_5\IPD\extensions_work\03_nfhs4_nfhs5_temporal\outputs\nfhs4_nfhs5_variable_crosswalk.csv


,harmonized_name,role,nfhs5_source,nfhs4_source,verified,harmonization_note
0,facility_type,exposure,"m15 (recoded: 10-13 home, 20-27 public, 30-33 ...",m15 (same recode function),True,DHS place-of-delivery coding is stable across ...
1,csection,outcome,m17,m17,True,Direct match.
2,respondent_id,design,caseid,caseid,True,Round-specific identifier; never pooled as if ...
3,cluster_number,design,v001,v001,True,Round-specific PSU; bootstrap resampling stays...
4,sample_stratum_v022,design,v022,v022,True,Round-specific sampling stratum.
5,sample_weight_normalized,design,v005 / 1e6,v005 / 1e6,True,Each round uses its own round-specific weight;...
6,birth_order,confounder,bord,bord,True,Direct match.
7,twin_order,confounder,b0,b0,True,Direct match.
8,wealth_index,confounder,v190,v190,True,"Direct match, but DHS wealth-index constructio..."
9,education_years,confounder,v133 (97 -> missing),v133 (97 -> missing),True,Direct match.


## 3. NFHS-4 raw-to-harmonized helper (adapt and run locally; verify before trusting)

This reproduces the exact `m15 -> facility_type` recoding used in `notebooks/01_data_exploration.ipynb`
(cell 7) and the exact special-code cleaning used for `s116`/`v133` in that same notebook, applied to a
raw NFHS-4 Birth Recode loaded with `pyreadstat` (or however you already load it). Run this function
once locally, save its output to `config.NFHS4_PROCESSED_DATA_PATH`, and Section 4 will pick it up from
there -- it is **not** executed against real data in this sandbox.

In [3]:
def recode_facility(code_value):
    """Identical recoding to notebooks/01_data_exploration.ipynb, cell 7."""
    if code_value in [10, 11, 12, 13]:
        return "home"
    elif 20 <= code_value <= 27:
        return "public"
    elif 30 <= code_value <= 33:
        return "private"
    else:
        return "other"


def harmonize_nfhs4_raw(df_raw):
    """Build the harmonized NFHS-4 analytic frame from a raw Birth Recode dataframe.

    df_raw must have the standard DHS mnemonic columns (caseid, v001, v005, v022, v024, v025,
    v130, v133, v190, bord, b0, s116, m15, m17) -- exactly as loaded from the .DTA/.sav file,
    unrenamed. Nothing here is executed against real data in this sandbox; this is the function
    a teammate runs locally against their authorized NFHS-4 copy.
    """
    required = ["caseid", "v001", "v005", "v022", "v024", "v025", "v130", "v133", "v190",
                "bord", "b0", "s116", "m15", "m17"]
    missing = [c for c in required if c not in df_raw.columns]
    assert not missing, (
        f"Raw NFHS-4 frame is missing required columns: {missing}. Confirm these exist under the "
        f"same mnemonics in your NFHS-4 extract before proceeding -- do not guess substitutes."
    )

    df = df_raw[df_raw["m15"].notna()].copy()
    df["facility_type"] = df["m15"].apply(recode_facility)
    df = df[df["facility_type"].isin(["public", "private", "other"])].copy()

    df["csection"] = df["m17"].astype(int)
    df["education_years"] = df["v133"].replace(97, np.nan)
    df["social_group"] = df["s116"].replace(8, np.nan)

    harmonized = pd.DataFrame({
        "respondent_id": df["caseid"],
        "cluster_number": df["v001"],
        "sample_weight": df["v005"],
        "sample_weight_normalized": df["v005"] / 1_000_000,
        "sample_stratum_v022": df["v022"],
        "state": df["v024"],
        "residence": df["v025"],
        "religion": df["v130"],
        "education_years": df["education_years"],
        "wealth_index": df["v190"],
        "birth_order": df["bord"],
        "twin_order": df["b0"],
        "social_group": df["social_group"],
        "facility_type": df["facility_type"],
        "csection": df["csection"],
    })
    harmonized["survey_round"] = "NFHS-4"
    return harmonized


print("harmonize_nfhs4_raw() defined. Example (run locally, not here):")
print("  import pyreadstat")
print("  raw, meta = pyreadstat.read_dta(str(config.NFHS4_BIRTH_RECODE_PATH))")
print("  harmonized = harmonize_nfhs4_raw(raw)")
print("  harmonized.to_parquet(config.NFHS4_PROCESSED_DATA_PATH, index=False)")

harmonize_nfhs4_raw() defined. Example (run locally, not here):
  import pyreadstat
  raw, meta = pyreadstat.read_dta(str(config.NFHS4_BIRTH_RECODE_PATH))
  harmonized = harmonize_nfhs4_raw(raw)
  harmonized.to_parquet(config.NFHS4_PROCESSED_DATA_PATH, index=False)


In [4]:
import pyreadstat

raw_nfhs4, meta_nfhs4 = pyreadstat.read_dta(str(config.NFHS4_BIRTH_RECODE_PATH))
print("Raw shape:", raw_nfhs4.shape)

harmonized_nfhs4 = harmonize_nfhs4_raw(raw_nfhs4)
print("Harmonized shape:", harmonized_nfhs4.shape)

harmonized_nfhs4.to_parquet(config.NFHS4_PROCESSED_DATA_PATH, index=False)
print("Saved to:", config.NFHS4_PROCESSED_DATA_PATH)

Raw shape: (1315617, 1340)


C:\Users\rifan\AppData\Local\Temp\ipykernel_58888\697303917.py:35: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df["social_group"] = df["s116"].replace(8, np.nan)


Harmonized shape: (195997, 16)
Saved to: C:\Users\rifan\Downloads\IPD_SEM_5\IPD\data\processed\df_model_nfhs4.parquet


## 4. Load both rounds

Loading fails loudly with an actionable message if the harmonized NFHS-4 file isn't present locally --
consistent with the rest of this project's "assert with a clear message" convention, rather than a bare
file-not-found traceback.

In [5]:
assert config.V2_PROCESSED_DATA_PATH.exists(), (
    f"{config.V2_PROCESSED_DATA_PATH} not found -- this is the same NFHS-5 file notebooks/v2/06-07 use."
)
assert config.NFHS4_PROCESSED_DATA_PATH.exists(), (
    f"{config.NFHS4_PROCESSED_DATA_PATH} not found. Run Section 3's harmonize_nfhs4_raw() against your "
    f"authorized local NFHS-4 Birth Recode and save its output to this path (or set the "
    f"IPD_NFHS4_PROCESSED_PATH environment variable) before running the rest of this notebook."
)

nfhs5, n_loaded_5, n_other_5 = utils.load_public_private_analytic(
    config.V2_PROCESSED_DATA_PATH,
    expected_total_rows=config.EXPECTED_V2_ROW_COUNT,
    expected_analytic_rows=config.EXPECTED_ANALYTIC_ROW_COUNT,
)
nfhs5["survey_round"] = "NFHS-5"

nfhs4, n_loaded_4, n_other_4 = utils.load_public_private_analytic(config.NFHS4_PROCESSED_DATA_PATH)
nfhs4["survey_round"] = "NFHS-4"

print(f"NFHS-5 analytic sample: {len(nfhs5)} (excluded 'other': {n_other_5})")
print(f"NFHS-4 analytic sample: {len(nfhs4)} (excluded 'other': {n_other_4})")

NFHS-5 analytic sample: 200794 (excluded 'other': 517)
NFHS-4 analytic sample: 195366 (excluded 'other': 631)


## 5. Harmonized confounder set -- full (primary) vs. reduced (required sensitivity, step 29)

The **same** confounder set must be used for both rounds (step 23). The primary set is the full
8-variable set only if every variable validates as present and non-degenerate in *both* rounds; any
variable flagged `verified = False` in the crosswalk, or found missing/constant in NFHS-4, is dropped
to build the reduced set used for the required harmonization sensitivity analysis (Section 10) -- this
choice is driven by the crosswalk's own verification flags, not picked after inspecting effect
estimates.

In [6]:
FULL_HARMONIZED_CONFOUNDERS = config.CONFOUNDER_COLS
unverified = set(variable_crosswalk.loc[~variable_crosswalk["verified"], "harmonized_name"])
unverified_confounders = unverified.intersection(FULL_HARMONIZED_CONFOUNDERS)

missing_or_degenerate = []
for col in FULL_HARMONIZED_CONFOUNDERS:
    if col not in nfhs4.columns:
        missing_or_degenerate.append(col)
    elif nfhs4[col].nunique(dropna=True) <= 1:
        missing_or_degenerate.append(col)

REDUCED_HARMONIZED_CONFOUNDERS = [
    c for c in FULL_HARMONIZED_CONFOUNDERS
    if c not in unverified_confounders and c not in missing_or_degenerate
]

print(f"Full harmonized confounder set ({len(FULL_HARMONIZED_CONFOUNDERS)}): {FULL_HARMONIZED_CONFOUNDERS}")
print(f"Dropped from the reduced set -- unverified in the crosswalk: {sorted(unverified_confounders)}")
print(f"Dropped from the reduced set -- missing/degenerate in NFHS-4: {sorted(set(missing_or_degenerate) - unverified_confounders)}")
print(f"Reduced (sensitivity) confounder set ({len(REDUCED_HARMONIZED_CONFOUNDERS)}): {REDUCED_HARMONIZED_CONFOUNDERS}")

PRIMARY_CONFOUNDERS = FULL_HARMONIZED_CONFOUNDERS if not unverified_confounders and not missing_or_degenerate \
    else REDUCED_HARMONIZED_CONFOUNDERS
print(f"\nPRIMARY_CONFOUNDERS used for the main round-specific AIPW analyses: {PRIMARY_CONFOUNDERS}")

PRIMARY_NUMERIC = [c for c in config.NUMERIC_CONFOUNDERS if c in PRIMARY_CONFOUNDERS]
PRIMARY_CATEGORICAL = [c for c in config.CATEGORICAL_CONFOUNDERS if c in PRIMARY_CONFOUNDERS]
REDUCED_NUMERIC = [c for c in config.NUMERIC_CONFOUNDERS if c in REDUCED_HARMONIZED_CONFOUNDERS]
REDUCED_CATEGORICAL = [c for c in config.CATEGORICAL_CONFOUNDERS if c in REDUCED_HARMONIZED_CONFOUNDERS]

Full harmonized confounder set (8): ['birth_order', 'wealth_index', 'education_years', 'residence', 'religion', 'social_group', 'twin_order', 'state']
Dropped from the reduced set -- unverified in the crosswalk: ['social_group', 'state']
Dropped from the reduced set -- missing/degenerate in NFHS-4: []
Reduced (sensitivity) confounder set (6): ['birth_order', 'wealth_index', 'education_years', 'residence', 'religion', 'twin_order']

PRIMARY_CONFOUNDERS used for the main round-specific AIPW analyses: ['birth_order', 'wealth_index', 'education_years', 'residence', 'religion', 'twin_order']


## 6. Round-specific survey-weighted descriptives (handoff step 25)

In [7]:
descriptive_rows = []
for round_name, df_round in [("NFHS-4", nfhs4), ("NFHS-5", nfhs5)]:
    for facility in ["public", "private"]:
        sub = df_round[df_round["facility_type"] == facility]
        raw_prev = sub["csection"].mean() * 100
        weighted_prev = np.average(sub["csection"], weights=sub["sample_weight_normalized"]) * 100
        descriptive_rows.append({
            "survey_round": round_name, "facility_type": facility, "n": len(sub),
            "raw_csection_prevalence_pct": raw_prev, "weighted_csection_prevalence_pct": weighted_prev,
        })

temporal_descriptives = pd.DataFrame(descriptive_rows)
temporal_descriptives.to_csv(OUTPUTS_DIR / "temporal_descriptives.csv", index=False)
print("Saved:", OUTPUTS_DIR / "temporal_descriptives.csv")
temporal_descriptives

Saved: C:\Users\rifan\Downloads\IPD_SEM_5\IPD\extensions_work\03_nfhs4_nfhs5_temporal\outputs\temporal_descriptives.csv


,survey_round,facility_type,n,raw_csection_prevalence_pct,weighted_csection_prevalence_pct
0,NFHS-4,public,141028,10.753184,11.917446
1,NFHS-4,private,54338,37.737863,40.854108
2,NFHS-5,public,150299,13.951523,14.312706
3,NFHS-5,private,50495,47.079909,47.365632


## 7. Round-specific overlap/balance diagnostics (handoff step 27)

Fits each round's own propensity model on `PRIMARY_CONFOUNDERS` (survey-weighted, no cross-fitting --
diagnostic only, exactly mirroring `notebooks/v2/06`'s convention) and reports standardized mean
differences before/after stabilized-IPW weighting, per round.

In [8]:
def round_balance_diagnostics(df_round, numeric_cols, categorical_cols):
    W = df_round[numeric_cols + categorical_cols].copy()
    for col in categorical_cols:
        W[col] = W[col].astype("string").fillna("Missing").astype(str)

    prop_pipeline = utils.make_propensity_pipeline(numeric_cols, categorical_cols, RANDOM_STATE)
    prop_pipeline.fit(W, df_round["exposure"], clf__sample_weight=df_round["sample_weight_normalized"])
    e_hat = np.clip(prop_pipeline.predict_proba(W)[:, 1], config.CLIP_EPS, 1 - config.CLIP_EPS)

    marginal_private = np.average(df_round["exposure"], weights=df_round["sample_weight_normalized"])
    stabilized_ipw = np.where(
        df_round["exposure"] == 1, marginal_private / e_hat, (1 - marginal_private) / (1 - e_hat)
    )
    combined_weight = df_round["sample_weight_normalized"].to_numpy() * stabilized_ipw

    balance_source = df_round[numeric_cols].copy()
    for col in numeric_cols:
        balance_source[col] = balance_source[col].fillna(balance_source[col].median())
    cat_source = df_round[categorical_cols].astype("string").fillna("Missing").astype(str)
    balance_dummies = pd.concat(
        [balance_source, pd.get_dummies(cat_source, columns=categorical_cols, prefix=categorical_cols)], axis=1
    )

    exposure_mask = df_round["exposure"].to_numpy() == 1
    unweighted_ones = np.ones(len(df_round))
    rows = []
    for col in balance_dummies.columns:
        if col in categorical_cols:
            continue
        values = balance_dummies[col].to_numpy(dtype=float)
        smd_before = utils.smd(values[exposure_mask], unweighted_ones[exposure_mask],
                                values[~exposure_mask], unweighted_ones[~exposure_mask])
        smd_after = utils.smd(values[exposure_mask], combined_weight[exposure_mask],
                               values[~exposure_mask], combined_weight[~exposure_mask])
        rows.append({"covariate": col, "smd_before": smd_before, "smd_after": smd_after})
    return pd.DataFrame(rows)


overlap_rows = []
for round_name, df_round in [("NFHS-4", nfhs4), ("NFHS-5", nfhs5)]:
    balance = round_balance_diagnostics(df_round, PRIMARY_NUMERIC, PRIMARY_CATEGORICAL)
    balance["survey_round"] = round_name
    overlap_rows.append(balance)

temporal_overlap_balance = pd.concat(overlap_rows, ignore_index=True)
temporal_overlap_balance.to_csv(OUTPUTS_DIR / "temporal_overlap_balance.csv", index=False)
print("Saved:", OUTPUTS_DIR / "temporal_overlap_balance.csv", "(supplementary diagnostic, not one of the five minimum required outputs)")
temporal_overlap_balance.groupby("survey_round")["smd_after"].apply(lambda s: (s.abs() > 0.1).sum())

Saved: C:\Users\rifan\Downloads\IPD_SEM_5\IPD\extensions_work\03_nfhs4_nfhs5_temporal\outputs\temporal_overlap_balance.csv (supplementary diagnostic, not one of the five minimum required outputs)


survey_round
NFHS-4    0
NFHS-5    0
Name: smd_after, dtype: int64

## 8. Round-specific cross-fitted AIPW (handoff steps 26/28)

Each round gets its **own** independent cross-fitted nuisance fit -- identical architecture to
`notebooks/v2/07` (respondent-grouped 5-fold cross-fitting, survey-weighted LogisticRegression
propensity + XGBoost S-learner outcome model, PSU-cluster bootstrap within stratum) -- rather than one
pooled model with a round interaction term. This is a deliberate design choice explained in Section 14.

In [9]:
result_nfhs4_primary = utils.run_cross_fitted_aipw(
    nfhs4, PRIMARY_CONFOUNDERS, PRIMARY_NUMERIC, PRIMARY_CATEGORICAL,
    n_folds=N_FOLDS, random_state=RANDOM_STATE, n_bootstrap=N_BOOTSTRAP, bootstrap_seed=config.BOOTSTRAP_SEED,
)
result_nfhs5_primary = utils.run_cross_fitted_aipw(
    nfhs5, PRIMARY_CONFOUNDERS, PRIMARY_NUMERIC, PRIMARY_CATEGORICAL,
    n_folds=N_FOLDS, random_state=RANDOM_STATE, n_bootstrap=N_BOOTSTRAP, bootstrap_seed=config.BOOTSTRAP_SEED,
)

for label, result in [("NFHS-4", result_nfhs4_primary), ("NFHS-5", result_nfhs5_primary)]:
    print(f"{label}: RD = {result['risk_difference']*100:.2f} pp "
          f"[{result['rd_ci'][0]*100:.2f}, {result['rd_ci'][1]*100:.2f}], "
          f"RR = {result['risk_ratio']:.3f} [{result['rr_ci'][0]:.3f}, {result['rr_ci'][1]:.3f}] "
          f"(n={result['n']})")

KeyboardInterrupt: 

## 9. Pooled time-by-sector contrast -- a directly estimated CI for the change

**Why a paired bootstrap, not a subtraction of two independently reported CIs.** NFHS-4 and NFHS-5 are
statistically independent samples: no respondent appears in both waves. For two independent estimators,
`Var(RD5_hat - RD4_hat) = Var(RD5_hat) + Var(RD4_hat)` exactly -- so pairing same-index bootstrap
replicates from each round's *already independently generated* replicate array
(`change_b = RD5_reps[b] - RD4_reps[b]`) produces a valid Monte Carlo sample from the true sampling
distribution of the change, and its percentiles are a formally correct CI for the change itself. This
is what the handoff means by "the CI for change is estimated directly," achieved here without needing
to fit a single mega-model with an explicit round interaction term (which would also need to somehow
represent counterfactual "what if this NFHS-4 respondent had been surveyed in round 5," which is not a
meaningful quantity -- round is observed per respondent, not manipulable).

In [ ]:
change_rd_reps = result_nfhs5_primary["rd_reps"] - result_nfhs4_primary["rd_reps"]
change_rd_point = result_nfhs5_primary["risk_difference"] - result_nfhs4_primary["risk_difference"]
change_rd_ci = (float(np.percentile(change_rd_reps, 2.5)), float(np.percentile(change_rd_reps, 97.5)))

change_rr_of_rr_reps = result_nfhs5_primary["rr_reps"] / result_nfhs4_primary["rr_reps"]
change_rr_of_rr_point = result_nfhs5_primary["risk_ratio"] / result_nfhs4_primary["risk_ratio"]
change_rr_of_rr_ci = (float(np.percentile(change_rr_of_rr_reps, 2.5)), float(np.percentile(change_rr_of_rr_reps, 97.5)))

print(f"Change in adjusted RD (NFHS-5 minus NFHS-4): {change_rd_point*100:.2f} pp "
      f"[{change_rd_ci[0]*100:.2f}, {change_rd_ci[1]*100:.2f}]")
print(f"Ratio of adjusted RRs (NFHS-5 / NFHS-4): {change_rr_of_rr_point:.3f} "
      f"[{change_rr_of_rr_ci[0]:.3f}, {change_rr_of_rr_ci[1]:.3f}]")

temporal_aipw_results = pd.DataFrame([
    {"survey_round": "NFHS-4", "confounder_set": "primary", "n": result_nfhs4_primary["n"],
     "risk_private_pct": result_nfhs4_primary["r1"] * 100, "risk_public_pct": result_nfhs4_primary["r0"] * 100,
     "risk_difference_pct": result_nfhs4_primary["risk_difference"] * 100,
     "risk_difference_ci_low_pct": result_nfhs4_primary["rd_ci"][0] * 100,
     "risk_difference_ci_high_pct": result_nfhs4_primary["rd_ci"][1] * 100,
     "risk_ratio": result_nfhs4_primary["risk_ratio"],
     "risk_ratio_ci_low": result_nfhs4_primary["rr_ci"][0], "risk_ratio_ci_high": result_nfhs4_primary["rr_ci"][1]},
    {"survey_round": "NFHS-5", "confounder_set": "primary", "n": result_nfhs5_primary["n"],
     "risk_private_pct": result_nfhs5_primary["r1"] * 100, "risk_public_pct": result_nfhs5_primary["r0"] * 100,
     "risk_difference_pct": result_nfhs5_primary["risk_difference"] * 100,
     "risk_difference_ci_low_pct": result_nfhs5_primary["rd_ci"][0] * 100,
     "risk_difference_ci_high_pct": result_nfhs5_primary["rd_ci"][1] * 100,
     "risk_ratio": result_nfhs5_primary["risk_ratio"],
     "risk_ratio_ci_low": result_nfhs5_primary["rr_ci"][0], "risk_ratio_ci_high": result_nfhs5_primary["rr_ci"][1]},
])
temporal_aipw_results.to_csv(OUTPUTS_DIR / "temporal_aipw_results.csv", index=False)
print("\nSaved:", OUTPUTS_DIR / "temporal_aipw_results.csv")
temporal_aipw_results

## 10. Harmonization sensitivity analysis -- reduced confounder set (handoff step 29)

In [ ]:
if PRIMARY_CONFOUNDERS == REDUCED_HARMONIZED_CONFOUNDERS:
    print("Primary and reduced confounder sets are identical -- no separate sensitivity re-fit needed; "
          "the primary result above already is the reduced-set result.")
    result_nfhs4_reduced, result_nfhs5_reduced = result_nfhs4_primary, result_nfhs5_primary
else:
    result_nfhs4_reduced = utils.run_cross_fitted_aipw(
        nfhs4, REDUCED_HARMONIZED_CONFOUNDERS, REDUCED_NUMERIC, REDUCED_CATEGORICAL,
        n_folds=N_FOLDS, random_state=RANDOM_STATE, n_bootstrap=N_BOOTSTRAP, bootstrap_seed=config.BOOTSTRAP_SEED,
    )
    result_nfhs5_reduced = utils.run_cross_fitted_aipw(
        nfhs5, REDUCED_HARMONIZED_CONFOUNDERS, REDUCED_NUMERIC, REDUCED_CATEGORICAL,
        n_folds=N_FOLDS, random_state=RANDOM_STATE, n_bootstrap=N_BOOTSTRAP, bootstrap_seed=config.BOOTSTRAP_SEED,
    )

change_rd_reps_reduced = result_nfhs5_reduced["rd_reps"] - result_nfhs4_reduced["rd_reps"]
change_rd_point_reduced = result_nfhs5_reduced["risk_difference"] - result_nfhs4_reduced["risk_difference"]
change_rd_ci_reduced = (float(np.percentile(change_rd_reps_reduced, 2.5)), float(np.percentile(change_rd_reps_reduced, 97.5)))

print(f"Sensitivity (reduced confounder set) change in adjusted RD: {change_rd_point_reduced*100:.2f} pp "
      f"[{change_rd_ci_reduced[0]*100:.2f}, {change_rd_ci_reduced[1]*100:.2f}]")

temporal_change_summary = pd.DataFrame([
    {"confounder_set": "primary", "confounders_used": ", ".join(PRIMARY_CONFOUNDERS),
     "rd_nfhs4_pct": result_nfhs4_primary["risk_difference"] * 100,
     "rd_nfhs5_pct": result_nfhs5_primary["risk_difference"] * 100,
     "change_in_rd_pct": change_rd_point * 100,
     "change_in_rd_ci_low_pct": change_rd_ci[0] * 100, "change_in_rd_ci_high_pct": change_rd_ci[1] * 100,
     "ratio_of_rr_nfhs5_over_nfhs4": change_rr_of_rr_point,
     "ratio_of_rr_ci_low": change_rr_of_rr_ci[0], "ratio_of_rr_ci_high": change_rr_of_rr_ci[1]},
    {"confounder_set": "sensitivity_reduced", "confounders_used": ", ".join(REDUCED_HARMONIZED_CONFOUNDERS),
     "rd_nfhs4_pct": result_nfhs4_reduced["risk_difference"] * 100,
     "rd_nfhs5_pct": result_nfhs5_reduced["risk_difference"] * 100,
     "change_in_rd_pct": change_rd_point_reduced * 100,
     "change_in_rd_ci_low_pct": change_rd_ci_reduced[0] * 100, "change_in_rd_ci_high_pct": change_rd_ci_reduced[1] * 100,
     "ratio_of_rr_nfhs5_over_nfhs4": np.nan, "ratio_of_rr_ci_low": np.nan, "ratio_of_rr_ci_high": np.nan},
])
temporal_change_summary.to_csv(OUTPUTS_DIR / "temporal_change_summary.csv", index=False)
print("\nSaved:", OUTPUTS_DIR / "temporal_change_summary.csv")
temporal_change_summary

## 11. QA / interpretation checks (explicit)

In [ ]:
print("Every pooled-model variable has the same substantive meaning across rounds:")
print(variable_crosswalk[["harmonized_name", "role", "verified"]].to_string(index=False))

print("\nRound-specific design handled separately (no pooled weight/PSU/stratum):",
      "sample_weight_normalized, cluster_number, and sample_stratum_v022 are each round's own -- "
      "confirmed by construction (Section 4 loads and bootstraps each round independently).")

print("\nFormal CI for the change in adjusted RD:",
      f"{change_rd_point*100:.2f} pp [{change_rd_ci[0]*100:.2f}, {change_rd_ci[1]*100:.2f}] -- reported, not omitted.")

print("\nCausal-language rule: a change in the adjusted sector gap between NFHS-4 and NFHS-5 describes "
      "how the cross-sectional adjusted association moved between two survey waves. It is NOT, by "
      "itself, causal evidence of any specific policy or programmatic effect -- no such design (e.g. a "
      "difference-in-differences with an identified policy shock) is implemented or claimed here.")

## 12. Figure -- temporal forest plot

In [ ]:
plot_rows = [
    ("NFHS-4", "primary", result_nfhs4_primary["risk_difference"] * 100,
     result_nfhs4_primary["rd_ci"][0] * 100, result_nfhs4_primary["rd_ci"][1] * 100),
    ("NFHS-5", "primary", result_nfhs5_primary["risk_difference"] * 100,
     result_nfhs5_primary["rd_ci"][0] * 100, result_nfhs5_primary["rd_ci"][1] * 100),
    ("Change (NFHS-5 minus NFHS-4)", "primary", change_rd_point * 100, change_rd_ci[0] * 100, change_rd_ci[1] * 100),
    ("NFHS-4", "sensitivity_reduced", result_nfhs4_reduced["risk_difference"] * 100,
     result_nfhs4_reduced["rd_ci"][0] * 100, result_nfhs4_reduced["rd_ci"][1] * 100),
    ("NFHS-5", "sensitivity_reduced", result_nfhs5_reduced["risk_difference"] * 100,
     result_nfhs5_reduced["rd_ci"][0] * 100, result_nfhs5_reduced["rd_ci"][1] * 100),
    ("Change (NFHS-5 minus NFHS-4)", "sensitivity_reduced", change_rd_point_reduced * 100,
     change_rd_ci_reduced[0] * 100, change_rd_ci_reduced[1] * 100),
]
plot_df = pd.DataFrame(plot_rows, columns=["label", "confounder_set", "estimate", "ci_low", "ci_high"])
plot_df["row_label"] = plot_df["label"] + " (" + plot_df["confounder_set"] + ")"

fig, ax = plt.subplots(figsize=(9, 6))
y_pos = np.arange(len(plot_df))
ax.errorbar(
    plot_df["estimate"], y_pos,
    xerr=[plot_df["estimate"] - plot_df["ci_low"], plot_df["ci_high"] - plot_df["estimate"]],
    fmt="o", color="steelblue", ecolor="gray", capsize=4,
)
ax.axvline(0, color="black", linestyle=":", linewidth=1)
ax.set_yticks(y_pos)
ax.set_yticklabels(plot_df["row_label"])
ax.invert_yaxis()
ax.set_xlabel("Adjusted risk difference, private minus public (percentage points)")
ax.set_title("NFHS-4 vs NFHS-5 Adjusted Sector Gap and Its Change, Primary vs Sensitivity Confounder Set")
fig.tight_layout()
fig.savefig(OUTPUTS_DIR / "temporal_forest_plot.png", dpi=200)
plt.show()
print("Saved:", OUTPUTS_DIR / "temporal_forest_plot.png")

## 13. Save metadata

In [ ]:
metadata = {
    "random_state": RANDOM_STATE,
    "n_nfhs4": result_nfhs4_primary["n"], "n_nfhs5": result_nfhs5_primary["n"],
    "primary_confounder_set": PRIMARY_CONFOUNDERS,
    "reduced_sensitivity_confounder_set": REDUCED_HARMONIZED_CONFOUNDERS,
    "dropped_unverified_confounders": sorted(unverified_confounders),
    "dropped_missing_or_degenerate_confounders": sorted(set(missing_or_degenerate) - unverified_confounders),
    "nuisance_models": "identical to notebooks/v2/07, fit independently per round (no pooled nuisance model)",
    "change_estimand": {
        "method": "paired PSU-cluster bootstrap difference: change_b = RD5_reps[b] - RD4_reps[b], "
                   "valid because NFHS-4 and NFHS-5 are independent samples",
        "n_bootstrap": N_BOOTSTRAP,
        "change_in_rd_pct": float(change_rd_point * 100),
        "change_in_rd_ci_pct": [change_rd_ci[0] * 100, change_rd_ci[1] * 100],
    },
    "interpretation_rule": (
        "A change in the adjusted sector gap between rounds is not causal evidence of a specific "
        "policy or programmatic effect; it describes how the cross-sectional adjusted association "
        "moved between two independent survey waves, each under its own round-specific measured-"
        "confounding assumption."
    ),
    "known_unresolved_harmonization_issues": [
        "state/UT boundary changes between 2015-16 and 2019-21 are not crosswalked to a common coding "
        "in this notebook -- state is excluded from the reduced confounder set until that crosswalk "
        "is built and verified.",
        "social_group (s116) coding in the NFHS-4 Birth Recode has not been confirmed identical to "
        "NFHS-5's -- excluded from the reduced confounder set until verified.",
        "wealth_index (v190) construction methodology has known cross-round revisions; treated as "
        "ordinal within-round only, not asserted as a continuously comparable scale across rounds.",
    ],
}

with open(OUTPUTS_DIR / "temporal_metadata.json", "w") as f:
    json.dump(metadata, f, indent=2, default=str)
print("Saved:", OUTPUTS_DIR / "temporal_metadata.json")

## 14. Documentation -- key design decisions and why

**Why separate round-specific nuisance models, not one pooled model with a round interaction.**
Facility-choice and C-section-risk relationships plausibly differ structurally between 2015-16 and
2019-21 (different private-sector penetration, different regulation, different obstetric norms) -- a
single pooled propensity/outcome model would implicitly assume those relationships are stable across
rounds except for a simple additive/multiplicative round term, which is a stronger and less
transparent assumption than letting each round's nuisance models be fully flexible on their own. Fitting
independently also sidesteps the awkward question of what a model conditioned on `survey_round` even
means for an individual respondent, who was only ever surveyed in one round.

**Why the change's CI comes from pairing independent bootstrap replicate arrays, not a delta-method
formula or a single subtraction of two reported CIs.** See Section 9's markdown -- because the two
samples are independent, `Var(diff) = Var(RD5) + Var(RD4)` exactly, and pairing same-index replicates
from each round's own already-computed bootstrap array is a standard, valid nonparametric way to draw
from the resulting joint distribution, without deriving or trusting a delta-method covariance formula.

**Why NFHS-4's confounder harmonization is deliberately conservative.** Two of the eight primary
confounders (`state`, `social_group`) carry documented, unresolved cross-round harmonization risk (state/
UT boundary changes; unconfirmed `s116` coding in NFHS-4). Rather than assume they harmonize cleanly,
this notebook automatically excludes any crosswalk-flagged or empirically degenerate variable from a
*reduced* confounder set and reports both the primary and reduced-set results side by side (Section 10),
so a reviewer sees exactly how sensitive the change estimate is to that specific harmonization choice --
this **is** the handoff's required harmonization sensitivity analysis (step 29), not a separate ad hoc
check.

**What this notebook does and does not establish.**
It estimates how the *adjusted association* between facility sector and Cesarean delivery moved between
two independent, cross-sectional NFHS waves, under each wave's own selection-on-observables assumption.
It does not identify a policy effect, does not imply the two waves' respondents are comparable beyond
the measured confounders, and does not resolve the state/social-group harmonization risk flagged above --
those remain explicitly open items for whoever integrates this extension into the main paper.